# FSDP / ZeRO Sharding from Scratch — Solution

## Setup

In [ ]:
!pip install torch --quiet

In [ ]:
import os
import math
import time
import copy
import torch
import torch.nn as nn
import torch.distributed as dist
import torch.multiprocessing as mp
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from dataclasses import dataclass
from functools import partial
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.distributed.fsdp import ShardingStrategy, MixedPrecision
from torch.distributed.fsdp.wrap import transformer_auto_wrap_policy

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
print(f"GPU count:       {torch.cuda.device_count()}")

WORLD_SIZE = max(torch.cuda.device_count(), 1)
USE_GPU    = torch.cuda.is_available()
print(f"\nSimulating {WORLD_SIZE} ranks  (GPU={'yes' if USE_GPU else 'no — CPU simulation'})")

### Toy model

In [ ]:
@dataclass
class ModelConfig:
    vocab_size:  int = 32000
    seq_len:     int = 512
    n_layers:    int = 6
    d_model:     int = 512
    n_heads:     int = 8
    d_ff:        int = 2048
    dropout:     float = 0.1

class CausalSelfAttention(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0
        self.n_heads = cfg.n_heads
        self.d_head  = cfg.d_model // cfg.n_heads
        self.qkv     = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=False)
        self.proj    = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.drop    = nn.Dropout(cfg.dropout)
        self.register_buffer(
            "mask",
            torch.tril(torch.ones(cfg.seq_len, cfg.seq_len)).view(1, 1, cfg.seq_len, cfg.seq_len)
        )

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).split(C, dim=-1)
        def reshape(t):
            return t.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        q, k, v = map(reshape, qkv)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = self.drop(torch.softmax(att, dim=-1))
        out = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out)

class TransformerBlock(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.ln1  = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg)
        self.ln2  = nn.LayerNorm(cfg.d_model)
        self.ff   = nn.Sequential(
            nn.Linear(cfg.d_model, cfg.d_ff),
            nn.GELU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.d_ff, cfg.d_model),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

class ToyGPT(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg    = cfg
        self.embed  = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos    = nn.Embedding(cfg.seq_len, cfg.d_model)
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layers)])
        self.ln_f   = nn.LayerNorm(cfg.d_model)
        self.head   = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

    def forward(self, idx):
        B, T = idx.shape
        pos  = torch.arange(T, device=idx.device)
        x    = self.embed(idx) + self.pos(pos)
        for block in self.blocks:
            x = block(x)
        return self.head(self.ln_f(x))

cfg      = ModelConfig()
model    = ToyGPT(cfg)
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,}  ({n_params/1e6:.1f}M)")

---
## Stage 1 — Memory Anatomy

In [ ]:
def bytes_to_mb(b: int) -> float:
    return b / (1024 ** 2)

def count_param_bytes(model: nn.Module, dtype=torch.float32) -> int:
    bytes_per_elem = torch.finfo(dtype).bits // 8
    return sum(p.numel() for p in model.parameters()) * bytes_per_elem

def estimate_adam_optimizer_bytes(model: nn.Module) -> int:
    """
    Adam stores momentum + variance per param, both fp32.
    = 2 states × 4 bytes × param_count
    """
    total_params = sum(p.numel() for p in model.parameters())
    return total_params * 2 * 4  # 2 states × fp32

def estimate_gradient_bytes(model: nn.Module, dtype=torch.float32) -> int:
    """
    Gradients match parameter shapes. fp32 for numerical stability.
    """
    bytes_per_elem = torch.finfo(dtype).bits // 8
    return sum(p.numel() for p in model.parameters()) * bytes_per_elem

def estimate_activation_bytes(
    cfg: ModelConfig,
    batch_size: int,
    dtype=torch.float32,
) -> int:
    """
    Approximation: n_layers × B × T × d_model × 34 × bytes_per_elem
    The constant 34 accounts for attention scores, residuals, FF intermediates.
    """
    bytes_per_elem = torch.finfo(dtype).bits // 8
    return cfg.n_layers * batch_size * cfg.seq_len * cfg.d_model * 34 * bytes_per_elem

def memory_breakdown(
    model: nn.Module,
    cfg: ModelConfig,
    batch_size: int = 4,
    param_dtype = torch.float32,
) -> dict:
    params_mb   = bytes_to_mb(count_param_bytes(model, param_dtype))
    grads_mb    = bytes_to_mb(estimate_gradient_bytes(model))   # grads stay fp32
    optim_mb    = bytes_to_mb(estimate_adam_optimizer_bytes(model))
    act_mb      = bytes_to_mb(estimate_activation_bytes(cfg, batch_size, param_dtype))
    total_mb    = params_mb + grads_mb + optim_mb + act_mb
    return {
        "params":      params_mb,
        "gradients":   grads_mb,
        "optimizer":   optim_mb,
        "activations": act_mb,
        "total":       total_mb,
    }

bd = memory_breakdown(model, cfg, batch_size=4)
for k, v in bd.items():
    print(f"{k:>12}: {v:>8.1f} MB")

expected_no_activation = n_params * 16 / (1024**2)
print(f"\n16 bytes/param rule: {expected_no_activation:.1f} MB")

In [ ]:
def plot_memory_breakdown(bd: dict, title="Memory breakdown (1 GPU, no sharding)"):
    keys   = ["params", "gradients", "optimizer", "activations"]
    values = [bd[k] for k in keys]
    colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]

    fig, ax = plt.subplots(figsize=(7, 4))
    bottom = 0
    for k, v, c in zip(keys, values, colors):
        ax.bar(0, v, bottom=bottom, color=c, alpha=0.85, width=0.4, label=k)
        if v > 5:
            ax.text(0, bottom + v / 2, f"{v:.0f} MB", ha="center", va="center",
                    fontsize=10, fontweight="bold", color="white")
        bottom += v

    ax.set_xlim(-0.5, 0.5)
    ax.set_xticks([])
    ax.set_ylabel("Memory (MB)")
    ax.set_title(f"{title}\nTotal: {bd['total']:.0f} MB")
    ax.legend(loc="upper right")
    plt.tight_layout()
    plt.show()

plot_memory_breakdown(bd)

---
## Stage 2 — Simulate ZeRO Stages Mathematically

In [ ]:
def zero_memory_per_rank(
    n_params:     int,
    world_size:   int,
    zero_stage:   int,
    param_dtype:  torch.dtype = torch.float32,
    batch_size:   int = 4,
    cfg:          ModelConfig = None,
) -> dict:
    N   = world_size
    bpe = torch.finfo(param_dtype).bits // 8

    # Full costs
    param_bytes = n_params * bpe
    grad_bytes  = n_params * 4          # always fp32
    optim_bytes = n_params * 8          # Adam: momentum + variance, fp32

    if zero_stage == 0:   # DDP: nothing sharded
        p_mb = bytes_to_mb(param_bytes)
        g_mb = bytes_to_mb(grad_bytes)
        o_mb = bytes_to_mb(optim_bytes)
    elif zero_stage == 1: # shard optimizer states
        p_mb = bytes_to_mb(param_bytes)
        g_mb = bytes_to_mb(grad_bytes)
        o_mb = bytes_to_mb(optim_bytes / N)
    elif zero_stage == 2: # shard optimizer + gradients
        p_mb = bytes_to_mb(param_bytes)
        g_mb = bytes_to_mb(grad_bytes / N)
        o_mb = bytes_to_mb(optim_bytes / N)
    elif zero_stage == 3: # shard everything (FSDP FULL_SHARD)
        p_mb = bytes_to_mb(param_bytes / N)
        g_mb = bytes_to_mb(grad_bytes / N)
        o_mb = bytes_to_mb(optim_bytes / N)
        # Temporary all-gather buffer during fwd/bwd (one layer at a time)
        # Approximation: ~1/N of total params in buffer
        p_mb += bytes_to_mb(param_bytes / N)
    else:
        raise ValueError(f"Invalid zero_stage: {zero_stage}")

    act_mb = 0.0
    if cfg is not None:
        act_mb = bytes_to_mb(estimate_activation_bytes(cfg, batch_size, param_dtype))

    return {
        "params_mb":      p_mb,
        "grads_mb":       g_mb,
        "optim_mb":       o_mb,
        "activations_mb": act_mb,
        "total_mb":       p_mb + g_mb + o_mb + act_mb,
    }

def compare_zero_stages(
    model: nn.Module,
    cfg:   ModelConfig,
    world_sizes: list = [1, 2, 4, 8, 16, 64],
    batch_size: int = 4,
) -> dict:
    n = sum(p.numel() for p in model.parameters())
    results = {}
    for N in world_sizes:
        results[N] = {}
        for stage in [0, 1, 2, 3]:
            results[N][stage] = zero_memory_per_rank(n, N, stage, batch_size=batch_size, cfg=cfg)
    return results

results     = compare_zero_stages(model, cfg)
world_sizes = [1, 2, 4, 8, 16, 64]

print(f"{'N':>4}  {'DDP(0)':>10}  {'ZeRO-1':>10}  {'ZeRO-2':>10}  {'ZeRO-3':>10}  (MB per GPU)")
print("-" * 58)
for N in world_sizes:
    row = [f"{results[N][s]['total_mb']:>10.1f}" for s in [0, 1, 2, 3]]
    print(f"{N:>4}  {'  '.join(row)}")

In [ ]:
def plot_zero_scaling(results: dict, gpu_memory_limit_gb: float = 40.0):
    world_sizes = sorted(results.keys())
    stages      = [0, 1, 2, 3]
    labels      = ["DDP (ZeRO-0)", "ZeRO-1", "ZeRO-2", "ZeRO-3 (FSDP)"]
    colors      = ["#C44E52", "#DD8452", "#4C72B0", "#55A868"]

    fig, ax = plt.subplots(figsize=(9, 5))
    for stage, label, color in zip(stages, labels, colors):
        mems = [results[N][stage]["total_mb"] for N in world_sizes]
        ax.plot(world_sizes, mems, "o-", label=label, color=color, linewidth=2)

    limit_mb = gpu_memory_limit_gb * 1024
    ax.axhline(limit_mb, linestyle="--", color="black", alpha=0.6,
               label=f"GPU limit ({gpu_memory_limit_gb:.0f} GB)")

    ax.set_xscale("log", base=2)
    ax.set_xlabel("World size (N GPUs)")
    ax.set_ylabel("Memory per GPU (MB)")
    ax.set_title("ZeRO stage memory scaling")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_zero_scaling(results)

### 2b — Communication primitives

In [ ]:
def simulate_all_reduce(tensors: list) -> list:
    """
    Sum all tensors, divide by N, return N copies.
    Communication: 2Ψ (ring all-reduce sends each byte twice).
    """
    N      = len(tensors)
    summed = sum(tensors)    # element-wise sum
    mean   = summed / N
    return [mean.clone() for _ in range(N)]

def simulate_reduce_scatter(tensors: list) -> list:
    """
    Sum all tensors element-wise, split into N equal chunks.
    Rank i receives chunk i.
    Communication: Ψ (half of all-reduce).
    """
    N      = len(tensors)
    summed = sum(tensors)
    chunks = summed.chunk(N)   # split into N equal pieces
    return [chunk.clone() for chunk in chunks]

def simulate_all_gather(shards: list) -> list:
    """
    Concatenate all shards, return N copies.
    Communication: Ψ.
    """
    N    = len(shards)
    full = torch.cat(shards, dim=0)
    return [full.clone() for _ in range(N)]

# Verify
N = 4; total_params = 1000
grads = [torch.ones(total_params) * (i + 1) for i in range(N)]

reduced = simulate_all_reduce(grads)
assert all(torch.allclose(r, torch.full((total_params,), 2.5)) for r in reduced)
print("✓ all-reduce")

scattered = simulate_reduce_scatter(grads)
assert torch.allclose(torch.cat(scattered), sum(grads))
print("✓ reduce-scatter")

shards   = [torch.arange(i*10, (i+1)*10, dtype=torch.float) for i in range(N)]
gathered = simulate_all_gather(shards)
assert all(torch.allclose(g, torch.arange(N*10, dtype=torch.float)) for g in gathered)
print("✓ all-gather")

elem_bytes = total_params * 4
print(f"\nCommunication volume (N={N}):")
print(f"  all-reduce:     {2*elem_bytes/1024:.1f} KB  (2Ψ — DDP)")
print(f"  reduce-scatter: {elem_bytes/1024:.1f} KB  (Ψ  — ZeRO-2)")
print(f"  all-gather:     {elem_bytes/1024:.1f} KB  (Ψ  — ZeRO-3 param recon)")

### 2c — ZeRO-3 layer simulation

In [ ]:
class ZeroThreeLayer:
    def __init__(self, in_features: int, out_features: int, rank: int, world_size: int):
        self.rank        = rank
        self.world_size  = world_size
        self.in_features  = in_features
        self.out_features = out_features
        self.shard_rows   = out_features // world_size

        full_weight = torch.randn(out_features, in_features)

        # Each rank owns rows [rank*shard_rows : (rank+1)*shard_rows]
        start = rank * self.shard_rows
        end   = start + self.shard_rows
        self.weight_shard = full_weight[start:end].clone()  # [shard_rows, in_features]
        self.grad_shard   = None

    def all_gather_weights(self, all_shards: list) -> torch.Tensor:
        """
        Collect weight shards from all ranks.
        Flatten → gather → reshape back to [out_features, in_features].
        """
        flat_shards = [s.flatten() for s in all_shards]
        gathered    = simulate_all_gather(flat_shards)
        full_flat   = gathered[self.rank]   # all ranks have same full tensor
        return full_flat.view(self.out_features, self.in_features)

    def forward(self, x: torch.Tensor, all_weight_shards: list):
        """
        FSDP-style forward:
          1. All-gather to reconstruct full weight
          2. Linear transform
          3. Full weight freed after this call (caller responsibility)
        """
        full_weight = self.all_gather_weights(all_weight_shards)  # temporary
        output      = x @ full_weight.T                           # [B, out_features]
        return output, full_weight

    def backward(self, x, full_weight, grad_out, all_grad_shards):
        """
        FSDP-style backward:
          1. Compute full weight gradient
          2. Reduce-scatter: rank i keeps shard i
          3. Compute input gradient
        """
        # Full grad w.r.t. weight: [out_features, in_features]
        grad_w_full = grad_out.T @ x                                           # [out, in]
        grad_w_flat = grad_w_full.flatten()

        # Simulate reduce-scatter — each rank holds 1/N of grad
        all_full_grads = [grad_w_flat.clone() for _ in range(self.world_size)]
        scattered_grads = simulate_reduce_scatter(all_full_grads)
        self.grad_shard = scattered_grads[self.rank].view(self.shard_rows, self.in_features)

        # grad w.r.t. input: [B, in_features]
        grad_x = grad_out @ full_weight
        return grad_x

# Verify
N = 2; in_f, out_f, batch = 8, 4, 3
layers = [ZeroThreeLayer(in_f, out_f, r, N) for r in range(N)]
x = torch.randn(batch, in_f)
all_shards = [l.weight_shard for l in layers]
out, full_w = layers[0].forward(x, all_shards)
print(f"Output shape:           {out.shape}")
print(f"Full weight shape:      {full_w.shape}")
grad_out = torch.randn_like(out)
grad_x   = layers[0].backward(x, full_w, grad_out, [None]*N)
print(f"grad_x shape:           {grad_x.shape}")
print(f"grad_shard shape:       {layers[0].grad_shard.shape}  (1/N of weight grad)")
shard_b = layers[0].weight_shard.numel() * 4
full_b  = out_f * in_f * 4
print(f"Memory ratio shard/full: {full_b/shard_b:.1f}x reduction")

---
## Stage 3 — DDP Baseline

In [ ]:
def setup_process_group(rank: int, world_size: int, backend: str = "nccl"):
    os.environ["MASTER_ADDR"] = "localhost"
    os.environ["MASTER_PORT"] = "12355"
    dist.init_process_group(backend, rank=rank, world_size=world_size)

def cleanup_process_group():
    dist.destroy_process_group()

def get_batch(batch_size, seq_len, vocab_size, device):
    x = torch.randint(0, vocab_size, (batch_size, seq_len), device=device)
    y = torch.randint(0, vocab_size, (batch_size, seq_len), device=device)
    return x, y

def ddp_train_fn(rank: int, world_size: int, results_queue, n_steps: int = 5):
    backend = "nccl" if USE_GPU else "gloo"
    setup_process_group(rank, world_size, backend)
    device = torch.device(f"cuda:{rank}") if USE_GPU else torch.device("cpu")

    # Build model, move to device, wrap with DDP
    m = ToyGPT(cfg).to(device)
    m = DDP(m, device_ids=[rank] if USE_GPU else None)

    optimizer = torch.optim.AdamW(m.parameters(), lr=1e-4)
    loss_fn   = nn.CrossEntropyLoss()

    if USE_GPU:
        torch.cuda.reset_peak_memory_stats(device)

    step_times = []
    for step in range(n_steps):
        t0 = time.time()

        x, y    = get_batch(2, cfg.seq_len, cfg.vocab_size, device)
        logits  = m(x)                                        # [B, T, vocab]
        loss    = loss_fn(logits.view(-1, cfg.vocab_size), y.view(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        step_times.append(time.time() - t0)

    peak_mb = 0.0
    if USE_GPU:
        peak_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2)

    if rank == 0:
        results_queue.put({
            "method":         "DDP",
            "world_size":     world_size,
            "mean_step_ms":   np.mean(step_times[1:]) * 1000,
            "peak_memory_mb": peak_mb,
        })
    cleanup_process_group()

In [ ]:
q = mp.Queue()
mp.spawn(ddp_train_fn, args=(WORLD_SIZE, q), nprocs=WORLD_SIZE, join=True)
ddp_result = q.get()
print("DDP:", ddp_result)

---
## Stage 4 — FSDP with PyTorch Native API

In [ ]:
def fsdp_train_fn(rank, world_size, results_queue, sharding_strategy, n_steps=5):
    backend = "nccl" if USE_GPU else "gloo"
    setup_process_group(rank, world_size, backend)
    device = torch.device(f"cuda:{rank}") if USE_GPU else torch.device("cpu")

    m = ToyGPT(cfg).to(device)

    # Wrap each TransformerBlock independently.
    # This is critical: FSDP all-gathers one unit at a time during fwd/bwd.
    # Wrapping the whole model as one unit would all-gather everything at once
    # and lose most of the memory benefit.
    wrap_policy = partial(
        transformer_auto_wrap_policy,
        transformer_layer_cls={TransformerBlock},
    )

    m = FSDP(
        m,
        auto_wrap_policy=wrap_policy,
        sharding_strategy=sharding_strategy,
        device_id=rank if USE_GPU else None,
    )

    optimizer = torch.optim.AdamW(m.parameters(), lr=1e-4)
    loss_fn   = nn.CrossEntropyLoss()

    if USE_GPU:
        torch.cuda.reset_peak_memory_stats(device)

    step_times = []
    for step in range(n_steps):
        t0 = time.time()

        x, y   = get_batch(2, cfg.seq_len, cfg.vocab_size, device)
        logits = m(x)
        loss   = loss_fn(logits.view(-1, cfg.vocab_size), y.view(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        step_times.append(time.time() - t0)

    peak_mb = 0.0
    if USE_GPU:
        peak_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2)

    if rank == 0:
        results_queue.put({
            "method":         f"FSDP/{sharding_strategy.name}",
            "world_size":     world_size,
            "mean_step_ms":   np.mean(step_times[1:]) * 1000,
            "peak_memory_mb": peak_mb,
        })
    cleanup_process_group()

In [ ]:
q = mp.Queue()
mp.spawn(fsdp_train_fn, args=(WORLD_SIZE, q, ShardingStrategy.FULL_SHARD), nprocs=WORLD_SIZE, join=True)
fsdp_full = q.get()
print("FSDP FULL_SHARD:", fsdp_full)

q = mp.Queue()
mp.spawn(fsdp_train_fn, args=(WORLD_SIZE, q, ShardingStrategy.SHARD_GRAD_OP), nprocs=WORLD_SIZE, join=True)
fsdp_grad = q.get()
print("FSDP SHARD_GRAD_OP:", fsdp_grad)

---
## Stage 5 — Memory & Communication Analysis

In [ ]:
all_results = [ddp_result, fsdp_full, fsdp_grad]

print(f"{'Method':<30}  {'Memory (MB)':>12}  {'Step (ms)':>10}")
print("-" * 58)
for r in all_results:
    print(f"{r['method']:<30}  {r['peak_memory_mb']:>12.1f}  {r['mean_step_ms']:>10.1f}")

In [ ]:
def plot_memory_vs_communication(results_list, simulated_results):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Left: measured peak memory per method
    methods = [r["method"] for r in results_list]
    mems    = [r["peak_memory_mb"] for r in results_list]
    colors  = ["#C44E52", "#55A868", "#4C72B0"]
    bars = ax1.bar(methods, mems, color=colors[:len(methods)], alpha=0.85)
    for bar, v in zip(bars, mems):
        ax1.text(bar.get_x() + bar.get_width()/2, v + 5,
                 f"{v:.0f}MB", ha="center", fontsize=9)
    ax1.set_ylabel("Peak GPU Memory (MB)")
    ax1.set_title("Measured peak memory per method")
    ax1.tick_params(axis='x', rotation=15)

    # Right: theoretical memory scaling from simulation
    world_sizes = sorted(simulated_results.keys())
    stage_labels = ["DDP", "ZeRO-1", "ZeRO-2", "ZeRO-3"]
    stage_colors = ["#C44E52", "#DD8452", "#4C72B0", "#55A868"]
    for s, label, color in zip([0,1,2,3], stage_labels, stage_colors):
        mems = [simulated_results[N][s]["total_mb"] for N in world_sizes]
        ax2.plot(world_sizes, mems, "o-", label=label, color=color, linewidth=2)
    ax2.set_xscale("log", base=2)
    ax2.axhline(40*1024, linestyle="--", color="black", alpha=0.5, label="40GB limit")
    ax2.set_xlabel("World size")
    ax2.set_ylabel("Memory per GPU (MB)")
    ax2.set_title("Theoretical memory scaling")
    ax2.legend(); ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

plot_memory_vs_communication(all_results, results)

In [ ]:
def communication_volume_analysis(n_params: int, world_size: int) -> dict:
    """
    Communication volume per training step.

    Ψ = n_params × 4 bytes  (fp32)

    DDP:    all-reduce grads           → 2Ψ
    ZeRO-1: reduce-scatter + updates   → 2Ψ  (same volume, different timing)
    ZeRO-2: reduce-scatter grads only  → Ψ
    ZeRO-3: reduce-scatter(Ψ) + 2×all-gather(2Ψ) = 3Ψ
             (all-gather happens once in fwd, once in bwd per layer)
    """
    psi = n_params * 4
    return {
        "ddp":    2 * psi,
        "zero1":  2 * psi,
        "zero2":  1 * psi,
        "zero3":  3 * psi,
    }

comm = communication_volume_analysis(n_params, WORLD_SIZE)
print("Communication volume per step:")
for method, vol in comm.items():
    print(f"  {method:>6}: {vol/1e9:.3f} GB")

print("\nKey insight: ZeRO-3 uses 3Ψ vs DDP's 2Ψ — more communication,")
print("but memory scales as 1/N. Worth it when model doesn't fit on 1 GPU.")

---
## Appendix — Why ZeRO-3 Communication is 3Ψ

```
Forward pass, for each layer L:
  all-gather(params_L)    →  Ψ_L bytes received
  compute activations
  free gathered params    ← memory freed immediately

Backward pass, for each layer L (reverse order):
  all-gather(params_L)    →  Ψ_L bytes again (needed for grad computation)
  compute gradients
  reduce-scatter(grads_L) →  Ψ_L bytes sent
  free gathered params

Total per step:
  2 × all-gather (fwd + bwd) = 2Ψ
  1 × reduce-scatter (bwd)   = Ψ
  ─────────────────────────────
  Total: 3Ψ
```

vs DDP:
```
  all-reduce gradients = 2Ψ  (ring: each byte sent once + received once)
```

ZeRO-3 pays 50% more in communication but achieves 1/N memory scaling.  
The crossover point depends on your interconnect bandwidth vs GPU memory size.

---
## Extension Exercises

**1. Mixed precision FSDP**  
`MixedPrecision(param_dtype=torch.bfloat16, reduce_dtype=torch.float32)`  
Params stored in bf16 (2 bytes) but gradients reduced in fp32.  
Memory for params halves; optimizer states stay fp32.

**2. Gradient checkpointing + FSDP**  
`torch.utils.checkpoint.checkpoint_wrapper(TransformerBlock(cfg))`  
Discards activations during forward, recomputes during backward.  
Cuts activation memory from O(layers × B × T × d) to O(√layers).

**3. CPU offloading**  
`FSDP(m, cpu_offload=CPUOffload(offload_params=True))`  
Params live on CPU, PCIe-transferred to GPU only when needed.  
Near-unlimited model size at severe throughput cost.

**4. Implement ZeRO-1 optimizer**  
Subclass `torch.optim.Optimizer`. In `step()`: update only your rank's shard,  
then call `dist.all_gather` to synchronize params across ranks.

**5. Find the GPU memory crossover**  
Scale up ModelConfig until DDP no longer fits on a 40GB GPU (simulated).  
Use the Stage 2 math to find minimum world_size for ZeRO-3 to fit.